# Entrevista

## Data Integration Engineer
- **Candidato**: TBD
- **Stack**: Python · DuckDB · S3 · API Banxico

### Objetivo
Construir desde cero un pipeline ETL productivo que consuma la **API SIE de Banxico**, transforme los datos con **DuckDB** y los persista en la carpeta **s3_datalake** en formato `Parquet`, con plena _observabilidad_ en cada capa.

![img](img/business_case.png)

### Criterios de evaluación
Se valora, en este orden:
1. **Idempotencia** — correr el pipeline dos veces produce el mismo estado final, sin duplicados.
2. **Manejo correcto del caso `UDIMXN.SPOT`** (ver sección dedicada más abajo).
3. **Justificación explícita de DuckDB vs Python** en cada transformación.
4. **Logging** estructurado en cada etapa (API → RAW → curada → estadísticas).
5. **Calidad del SQL y del modelo de datos** (claves naturales, columnas de auditoría, nombres consistentes).

### Notas sobre la API de Banxico
- `fecha` viene como string `DD/MM/YYYY` y `dato` como string — el casting queda en el código del candidato, no en el cliente.
- La metadata de cada serie trae `fechaInicio` y `fechaFin`; esta última **no** siempre coincide con la fecha del último valor *actual* (ver UDIS).

### Security List
Considere la siguiente lista de securities. El `IDSERIE` a usar con la API se resuelve desde la tabla `BANXICO_SERIES` de la base local.

| SECURITY_NAME | Periodicidad efectiva            |
|---------------|----------------------------------|
| `USDMXN.FIX`  | Diaria hábil                     |
| `UDIMXN.SPOT` | Diaria, publicada en bloques (*) |
| `MXN.TPFB`    | Diaria hábil                     |
| `MXN.TIIE-1D` | Diaria hábil                     |

(*) Detalle en la sección **Caso especial UDIMXN.SPOT**.

In [2]:
from vmetrix import get_database

security_list = [
    "USDMXN.FIX",
    "UDIMXN.SPOT",
    "MXN.TPFB",
    "MXN.TIIE-1D"
]

security_list_str = ", ".join(f"'{sec}'" for sec in security_list)

db = get_database()

sql = f"""
select *
from BANXICO_SERIES
where SECURITY_NAME IN ({security_list_str})
"""

db.query(sql)

,SECURITY_NAME,IDSERIE,TITULO,PERIODICIDAD,CIFRA,UNIDAD
0,USDMXN.FIX,SF43718,Tipo de cambio ...,Diaria,Tipo de Cambio,Pesos por Dólar
1,MXN.TIIE-1D,SF331451,"TIIE de Fondeo a Un Día Hábil Bancario, Median...",Diaria,Tasas Promedio,Porcentajes
2,UDIMXN.SPOT,SP68257,Valor de UDIS,Diaria,Tipo de Cambio,Unidades de Inversión
3,MXN.TPFB,SF43773,Tasa de fondeo bancario Mediana ponderada por ...,Diaria,Porcentajes,Sin Unidad


### 1. Carga histórica

Cargue los securities indicados desde **2025-01-01 hasta hoy** en LocalDb.

**Entregables**
1. Documantación y DDL en [vmetrix/database.sql](vmetrix/database.sql) con la(s) definicion(es) de la(s) tabla(s) que diseñe.
2. Creación efectiva de esas tablas en `LocalDb`.
3. Script Python que ejecute la carga.

In [ ]:
# Setup de logging a archivo para toda la corrida del pipeline.
# - Mantiene el StreamHandler ya configurado en vmetrix/__init__.py (consola).
# - Adjunta un FileHandler al logger raíz "vmetrix" -> captura logs de todos los
#   submódulos (vmetrix.banxico_api, vmetrix.raw, etc.) por propagación.
# - Un archivo por corrida en ./logs/pipeline_<timestamp>.log para auditoría.
# - Idempotente: si la celda se re-ejecuta, reemplaza el FileHandler en lugar de duplicar.
import logging
from datetime import datetime
from pathlib import Path

LOGS_DIR = Path("logs")
LOGS_DIR.mkdir(exist_ok=True)

run_ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_path = LOGS_DIR / f"pipeline_{run_ts}.log"

vmetrix_logger = logging.getLogger("vmetrix")

# Quitar FileHandlers previos (evita duplicar líneas si re-ejecutas la celda)
for h in list(vmetrix_logger.handlers):
    if isinstance(h, logging.FileHandler):
        vmetrix_logger.removeHandler(h)
        h.close()

file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter(
    fmt="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
))
vmetrix_logger.addHandler(file_handler)

vmetrix_logger.info("=" * 60)
vmetrix_logger.info("Inicio de corrida del pipeline | log: %s", log_path)
vmetrix_logger.info("=" * 60)

log_path

In [ ]:
# Bootstrap del esquema curado: crea BANXICO_VALUES si aún no existe en db.duckdb.
# Idempotente — seguro de re-ejecutar en cada corrida del pipeline.
from vmetrix import get_database

db = get_database()
db.command("""
    CREATE TABLE IF NOT EXISTS BANXICO_VALUES (
      SECURITY_NAME   VARCHAR     NOT NULL,
      IDSERIE         VARCHAR     NOT NULL,
      FECHA           DATE        NOT NULL,
      VALOR           DECIMAL(18, 8),
      EXECUTION_DATE  DATE        NOT NULL,
      INGESTED_AT     TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
      SOURCE_FILE     VARCHAR,
      API_FECHA_RAW   VARCHAR,
      PRIMARY KEY (IDSERIE, FECHA)
    )
""")

db.query("DESCRIBE BANXICO_VALUES")

In [ ]:
from datetime import date
from vmetrix import get_database, get_banxico_api

# --- Parámetros de ejecución ---
execution_date = date.today()           # día de ejecución del pipeline (siempre hoy)
start_date = date(2025, 1, 1)           # límite inferior fijo del histórico
end_date = execution_date               # default = hoy; sobrescribir para backfills puntuales

# --- 1. Resolver IDSERIE desde BANXICO_SERIES ---
security_list = ["USDMXN.FIX", "UDIMXN.SPOT", "MXN.TPFB", "MXN.TIIE-1D"]
security_list_str = ", ".join(f"'{s}'" for s in security_list)

db = get_database()
series_df = db.query(f"""
    SELECT SECURITY_NAME, IDSERIE
    FROM BANXICO_SERIES
    WHERE SECURITY_NAME IN ({security_list_str})
""")

# Mapa {SECURITY_NAME: IDSERIE} y string CSV para la API (una sola llamada multi-serie)
id_map = dict(zip(series_df["SECURITY_NAME"], series_df["IDSERIE"]))
series_csv = ",".join(id_map.values())

# --- 2. Llamada a get_values_between (la API espera YYYY-MM-DD) ---
api = get_banxico_api()
raw_json = api.get_values_between(
    series=series_csv,
    start_date=start_date.isoformat(),
    end_date=end_date.isoformat(),
)

raw_json

In [ ]:
# RAW -> Parquet (capa bronze) en s3_datalake/.
# Estrategia: el response completo de la API se guarda como un BLOB JSON en una sola
# fila Parquet — fidelidad total (no se descarta ningún campo del response, a diferencia
# de un aplanado tabular). Cumple con el requerimiento del brief de Parquet, y conserva
# la semántica "raw" más estricta.
#
# Decisión DuckDB vs Python:
#   - Serialización del dict a JSON string: Python (json.dumps — operación trivial).
#   - Escritura Parquet: DuckDB (zero-copy desde DataFrame, compresión zstd).
#
# El aplanado a tabla queryable se hará en el siguiente paso (curated -> BANXICO_VALUES)
# usando read_json_auto + UNNEST directamente desde DuckDB sobre este Parquet.
import json as _json
from datetime import datetime
from pathlib import Path
import logging

import pandas as pd
import duckdb

logger = logging.getLogger("vmetrix.raw")

# --- 1. Empaquetar response como BLOB JSON + metadata de ingestión ---
fetched_at = datetime.now()
n_series = len(raw_json.get("bmx", {}).get("series", []))
n_datos = sum(len(s.get("datos", [])) for s in raw_json.get("bmx", {}).get("series", []))

raw_df = pd.DataFrame([{
    "endpoint": "datos_range",
    "window_start": start_date.isoformat(),
    "window_end": end_date.isoformat(),
    "execution_date": execution_date.isoformat(),
    "fetched_at": fetched_at,
    "n_series": n_series,         # auditoría rápida sin tener que parsear el payload
    "n_datos": n_datos,
    "payload": _json.dumps(raw_json, ensure_ascii=False),  # response íntegro
}])

logger.info(
    "RAW empaquetado: %d series, %d datos, payload=%d bytes",
    n_series, n_datos, len(raw_df.iloc[0]["payload"]),
)

# --- 2. Construir ruta Hive-partitioned ---
DATALAKE = Path("s3_datalake")
partition_dir = (
    DATALAKE
    / "raw" / "banxico_sie"
    / "endpoint=datos_range"
    / f"execution_date={execution_date.isoformat()}"
)
partition_dir.mkdir(parents=True, exist_ok=True)

filename = f"window_{start_date.isoformat()}_{end_date.isoformat()}.parquet"
parquet_path = partition_dir / filename

# Idempotencia a nivel de archivo: si existe, lo sobrescribimos.
if parquet_path.exists():
    logger.info("Sobrescribiendo Parquet existente: %s", parquet_path)

# --- 3. Escribir Parquet (DuckDB in-memory: no toca db.duckdb) ---
with duckdb.connect() as conn:
    conn.register("_raw_df", raw_df)
    conn.execute(
        f"COPY _raw_df TO '{parquet_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
    )

logger.info("RAW persistido en %s", parquet_path)
parquet_path

In [ ]:
# Capa curada: RAW Parquet -> BANXICO_VALUES (parse + cast + idempotencia).
#
# Decisión DuckDB vs Python (en cada transformación):
#   - Parse JSON del payload          : DuckDB (from_json con schema -> struct nativo).
#   - UNNEST series y datos           : DuckDB (operación natural sobre LIST/STRUCT).
#   - Cast fecha DD/MM/YYYY -> DATE   : DuckDB (STRPTIME, vectorizado).
#   - Cast dato str -> DECIMAL(18,8)  : DuckDB (TRY_CAST: NULL si "N/E", sin try/except).
#   - Join con BANXICO_SERIES         : DuckDB (los datos viven ahí; round-trip a Python sería tonto).
#   - DELETE+INSERT idempotente       : DuckDB (DML transaccional).
#   - Python                          : solo orquestación (paths, parámetros, logging).
import logging

logger = logging.getLogger("vmetrix.curated")

# Schema esperado del payload (lo que devuelve /datos/{start}/{end}).
# Definirlo explícitamente da: (a) errores tempranos si la API cambia,
# (b) acceso por struct.field (más legible que json_extract).
PAYLOAD_SCHEMA = (
    '{"bmx":{"series":[{"idSerie":"VARCHAR","titulo":"VARCHAR",'
    '"datos":[{"fecha":"VARCHAR","dato":"VARCHAR"}]}]}}'
)

raw_glob = "s3_datalake/raw/banxico_sie/endpoint=datos_range/**/*.parquet"
ids_csv = ", ".join(f"'{i}'" for i in id_map.values())

db = get_database()

# --- 1. Idempotencia: borrar registros previos para esta ventana e IDSERIEs ---
# Patrón DELETE+INSERT (no UPSERT) porque si la API devuelve menos fechas que la
# corrida anterior (raro pero posible), no queremos dejar filas huérfanas.
n_before = int(db.query(f"""
    SELECT COUNT(*) AS n FROM BANXICO_VALUES
    WHERE IDSERIE IN ({ids_csv})
      AND FECHA BETWEEN DATE '{start_date}' AND DATE '{end_date}'
""").iloc[0]["n"])
logger.info(
    "Idempotencia: %d filas existentes serán reemplazadas para [%s, %s]",
    n_before, start_date, end_date,
)

db.command(f"""
    DELETE FROM BANXICO_VALUES
    WHERE IDSERIE IN ({ids_csv})
      AND FECHA BETWEEN DATE '{start_date}' AND DATE '{end_date}'
""")

# --- 2. INSERT desde Parquet RAW: parse + unnest + cast + join, todo en una query ---
db.command(f"""
    INSERT INTO BANXICO_VALUES
        (SECURITY_NAME, IDSERIE, FECHA, VALOR, EXECUTION_DATE, SOURCE_FILE, API_FECHA_RAW)
    WITH parsed AS (
        SELECT
            from_json(payload, '{PAYLOAD_SCHEMA}') AS j,
            execution_date,
            filename
        FROM read_parquet('{raw_glob}', filename=true)
        WHERE execution_date = '{execution_date}'
    ),
    series AS (
        SELECT execution_date, filename,
               UNNEST(j.bmx.series) AS s
        FROM parsed
    ),
    flat AS (
        SELECT execution_date, filename,
               s.idSerie AS idserie,
               UNNEST(s.datos) AS d
        FROM series
    )
    SELECT
        bs.SECURITY_NAME                                AS security_name,
        flat.idserie                                    AS idserie,
        STRPTIME(d.fecha, '%d/%m/%Y')::DATE             AS fecha,
        TRY_CAST(d.dato AS DECIMAL(18, 8))              AS valor,  -- NULL si "N/E"
        CAST(flat.execution_date AS DATE)               AS execution_date,
        flat.filename                                   AS source_file,
        d.fecha                                         AS api_fecha_raw
    FROM flat
    LEFT JOIN BANXICO_SERIES bs ON bs.IDSERIE = flat.idserie
""")

# --- 3. Verificación: resumen por security ---
result = db.query(f"""
    SELECT SECURITY_NAME,
           COUNT(*)                                            AS n_filas,
           MIN(FECHA)                                          AS f_min,
           MAX(FECHA)                                          AS f_max,
           SUM(CASE WHEN VALOR IS NULL THEN 1 ELSE 0 END)      AS n_null,
           ROUND(AVG(VALOR), 4)                                AS avg_valor
    FROM BANXICO_VALUES
    WHERE EXECUTION_DATE = DATE '{execution_date}'
    GROUP BY SECURITY_NAME
    ORDER BY SECURITY_NAME
""")
logger.info("Carga curated completada: %d securities en BANXICO_VALUES", len(result))
result

# --- 4. Silver export (opción B): materializar el snapshot curado a Parquet ---
# Justificación: BANXICO_VALUES en DuckDB ya es la "gold". Exportar a Parquet
# silver da: (a) portabilidad para otros engines, (b) backup desacoplado de db.duckdb,
# (c) snapshot por execution_date+load_type para auditar/comparar corridas.
# Es DuckDB nativo (COPY) — sin Python en el medio.
silver_dir = (
    DATALAKE / "silver" / "banxico_sie" / "values"
    / f"execution_date={execution_date.isoformat()}"
)
silver_dir.mkdir(parents=True, exist_ok=True)
silver_path = silver_dir / "load=historical.parquet"

db.command(f"""
    COPY (
        SELECT * FROM BANXICO_VALUES
        WHERE EXECUTION_DATE = DATE '{execution_date}'
          AND IDSERIE IN ({ids_csv})
          AND FECHA BETWEEN DATE '{start_date}' AND DATE '{end_date}'
    ) TO '{silver_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
logger.info("Silver histórico exportado: %s", silver_path)


### 2. Carga diaria incremental

Implemente la arquitectura descrita en la imagen anterior para que el pipeline pueda correr todos los días como un proceso diario.

**2.1 Valores del día**
- Traer los valores vigentes al día de hoy para los cuatro securities.
- Para `UDIMXN.SPOT` ver el caso especial abajo (**NO** se asume un solo valor diario).

**2.2 Estadísticas móviles (7 días)**
Cree una tabla de estadísticas de los últimos **7 días calendario** (ventana `[value_date - 6, value_date]`)
- Evalúe si es necesario una tabla, vista o función para resolver este problema.
    - Justifique su respuesta.

Se usan **7 días calendario** (no hábiles) para que la definición sea la misma para todos los securities

In [ ]:
# Carga incremental — Paso 1: get_last_value + RAW bronze
#
# get_last_value('SF43718,SP68257,SF331451,SF43773') devuelve UNA fila por serie con
# la fecha más reciente publicada. Importante:
#   - Series diaria hábil: la fecha es el último día hábil cerrado (puede ser ayer).
#   - UDIMXN.SPOT: la fecha puede estar en el FUTURO (horizonte publicado en bloques).
# El idempotency key del incremental es (IDSERIE, FECHA-de-la-API), no "today".
import json as _json
from datetime import datetime
from pathlib import Path
import logging

import pandas as pd
import duckdb

logger = logging.getLogger("vmetrix.incremental")

# --- 1. Llamada a la API ---
api = get_banxico_api()
incr_raw_json = api.get_last_value(series=series_csv)

# --- 2. Logger por security: qué fecha y valor trajo la API ---
inv_id_map = {v: k for k, v in id_map.items()}
for s in incr_raw_json["bmx"]["series"]:
    idserie = s["idSerie"]
    sec = inv_id_map.get(idserie, idserie)
    if s.get("datos"):
        d = s["datos"][0]
        logger.info(
            "get_last_value | %-12s (%s) -> fecha=%s, dato=%s",
            sec, idserie, d["fecha"], d["dato"],
        )
    else:
        logger.warning("get_last_value | %s (%s) -> sin datos", sec, idserie)

# --- 3. Persistir RAW bronze (mismo patrón que histórico, distinta partición Hive) ---
fetched_at_incr = datetime.now()
n_series_incr = len(incr_raw_json.get("bmx", {}).get("series", []))
n_datos_incr = sum(
    len(s.get("datos", [])) for s in incr_raw_json.get("bmx", {}).get("series", [])
)

incr_raw_df = pd.DataFrame([{
    "endpoint": "oportuno",
    "execution_date": execution_date.isoformat(),
    "fetched_at": fetched_at_incr,
    "n_series": n_series_incr,
    "n_datos": n_datos_incr,
    "payload": _json.dumps(incr_raw_json, ensure_ascii=False),
}])

incr_partition_dir = (
    DATALAKE / "raw" / "banxico_sie"
    / "endpoint=oportuno"
    / f"execution_date={execution_date.isoformat()}"
)
incr_partition_dir.mkdir(parents=True, exist_ok=True)
incr_parquet_path = incr_partition_dir / "last_value.parquet"

if incr_parquet_path.exists():
    logger.info("Sobrescribiendo Parquet bronze incremental: %s", incr_parquet_path)

with duckdb.connect() as conn:
    conn.register("_incr_raw_df", incr_raw_df)
    conn.execute(
        f"COPY _incr_raw_df TO '{incr_parquet_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
    )

logger.info("RAW incremental persistido: %s", incr_parquet_path)
incr_raw_json

In [ ]:
# Carga incremental — Paso 2: gap detection + backfill por serie + UPSERT + silver
#
# Patrón: el incremental REAL no es "tomar el último valor de la API" sino
# "asegurarme de tener todas las fechas entre lo último cargado y lo último publicado".
# Esto subsume tres escenarios con una sola lógica:
#   1) UDIS: api_last es futura -> backfill multi-día.
#   2) Pipeline corre el lunes después de festivo -> backfill viernes/sáb/dom.
#   3) Pipeline al día -> NO_GAP, no-op limpio.
#
# Decisión DuckDB vs Python (por paso):
#   - Cálculo de gaps (api_last vs MAX(FECHA) por serie): DuckDB (un join trivial).
#   - HTTP get_values_between por serie con gap                : Python (no hay HTTP en SQL).
#   - Parse JSON + UNNEST + cast + UPSERT desde RAW            : DuckDB.
#   - Silver export                                            : DuckDB COPY.
#   - Loop de orquestación + logging                           : Python.
import logging
import json as _json
from datetime import datetime
from pathlib import Path

import pandas as pd
import duckdb

logger = logging.getLogger("vmetrix.incremental.gapfill")

INCR_OPORTUNO_GLOB  = "s3_datalake/raw/banxico_sie/endpoint=oportuno/**/*.parquet"
INCR_GAPFILL_GLOB   = "s3_datalake/raw/banxico_sie/endpoint=datos_range_incremental/**/*.parquet"

PAYLOAD_SCHEMA = (
    '{"bmx":{"series":[{"idSerie":"VARCHAR","titulo":"VARCHAR",'
    '"datos":[{"fecha":"VARCHAR","dato":"VARCHAR"}]}]}}'
)
ids_csv_incr = ", ".join(f"'{i}'" for i in id_map.values())

db = get_database()

# --- 1. Calcular gaps por serie [last_db + 1, api_last] ---
gaps_df = db.query(f"""
    WITH parsed AS (
        SELECT from_json(payload, '{PAYLOAD_SCHEMA}') AS j
        FROM read_parquet('{INCR_OPORTUNO_GLOB}')
        WHERE execution_date = '{execution_date}'
    ),
    series AS (
        SELECT UNNEST(j.bmx.series) AS s FROM parsed
    ),
    datos AS (
        SELECT s.idSerie AS idserie, UNNEST(s.datos) AS d FROM series
    ),
    api_last_per_serie AS (
        SELECT idserie,
               MAX(STRPTIME(d.fecha, '%d/%m/%Y')::DATE) AS api_last_fecha
        FROM datos
        GROUP BY idserie
    ),
    db_last_per_serie AS (
        SELECT IDSERIE AS idserie, MAX(FECHA) AS db_last_fecha
        FROM BANXICO_VALUES
        WHERE IDSERIE IN ({ids_csv_incr})
        GROUP BY IDSERIE
    )
    SELECT
        a.idserie,
        a.api_last_fecha,
        b.db_last_fecha,
        CAST(COALESCE(b.db_last_fecha, DATE '{start_date}' - INTERVAL 1 DAY) + INTERVAL 1 DAY AS DATE) AS fetch_start,
        a.api_last_fecha AS fetch_end,
        CASE
            WHEN b.db_last_fecha IS NULL                     THEN 'INIT'
            WHEN a.api_last_fecha > b.db_last_fecha          THEN 'GAP'
            ELSE 'NO_GAP'
        END AS status
    FROM api_last_per_serie a
    LEFT JOIN db_last_per_serie b ON a.idserie = b.idserie
    ORDER BY a.idserie
""")

logger.info("Análisis de gaps por serie:\n%s", gaps_df.to_string(index=False))

# --- 2. Por cada serie con gap real, llamar get_values_between y persistir RAW ---
gap_files = []
for row in gaps_df.itertuples(index=False):
    sec = next((k for k, v in id_map.items() if v == row.idserie), row.idserie)
    if row.status == "NO_GAP" or row.fetch_start > row.fetch_end:
        logger.info(
            "Gap-fill | %-12s (%s): sin gap (db=%s, api=%s)",
            sec, row.idserie, row.db_last_fecha, row.api_last_fecha,
        )
        continue

    fs = pd.Timestamp(row.fetch_start).strftime("%Y-%m-%d")
    fe = pd.Timestamp(row.fetch_end).strftime("%Y-%m-%d")
    logger.info("Gap-fill | %-12s (%s): solicitando rango [%s, %s]", sec, row.idserie, fs, fe)

    payload = api.get_values_between(series=row.idserie, start_date=fs, end_date=fe)
    n_datos_serie = sum(len(s.get("datos", [])) for s in payload.get("bmx", {}).get("series", []))

    fetched_at_gap = datetime.now()
    raw_df_gap = pd.DataFrame([{
        "endpoint": "datos_range_incremental",
        "idserie": row.idserie,
        "window_start": fs,
        "window_end": fe,
        "execution_date": execution_date.isoformat(),
        "fetched_at": fetched_at_gap,
        "n_datos": n_datos_serie,
        "payload": _json.dumps(payload, ensure_ascii=False),
    }])

    gap_dir = (
        DATALAKE / "raw" / "banxico_sie"
        / "endpoint=datos_range_incremental"
        / f"execution_date={execution_date.isoformat()}"
    )
    gap_dir.mkdir(parents=True, exist_ok=True)
    gap_path = gap_dir / f"idserie={row.idserie}_{fs}_{fe}.parquet"

    with duckdb.connect() as conn:
        conn.register("_gap_df", raw_df_gap)
        conn.execute(
            f"COPY _gap_df TO '{gap_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
        )

    logger.info("Gap-fill | %s: RAW guardado %s (%d datos)", row.idserie, gap_path.name, n_datos_serie)
    gap_files.append(str(gap_path))

# --- 3. UPSERT desde TODOS los RAW de gap-fill recién escritos (si los hubo) ---
if not gap_files:
    logger.info("Incremental: sin gaps que cubrir. Pipeline al día.")
    summary = db.query(f"""
        SELECT bv.SECURITY_NAME, bv.IDSERIE,
               MAX(bv.FECHA) AS ULTIMA_FECHA,
               MAX(bv.INGESTED_AT) AS LAST_UPDATE
        FROM BANXICO_VALUES bv
        WHERE bv.IDSERIE IN ({ids_csv_incr})
        GROUP BY bv.SECURITY_NAME, bv.IDSERIE
        ORDER BY bv.SECURITY_NAME
    """)
else:
    db.command(f"""
        CREATE OR REPLACE TABLE _BANXICO_VALUES_GAP_STAGE AS
        WITH parsed AS (
            SELECT from_json(payload, '{PAYLOAD_SCHEMA}') AS j,
                   execution_date,
                   filename
            FROM read_parquet('{INCR_GAPFILL_GLOB}', filename=true)
            WHERE execution_date = '{execution_date}'
        ),
        series AS (
            SELECT execution_date, filename, UNNEST(j.bmx.series) AS s
            FROM parsed
        ),
        flat AS (
            SELECT execution_date, filename,
                   s.idSerie AS idserie,
                   UNNEST(s.datos) AS d
            FROM series
        )
        SELECT
            bs.SECURITY_NAME                              AS SECURITY_NAME,
            flat.idserie                                  AS IDSERIE,
            STRPTIME(d.fecha, '%d/%m/%Y')::DATE           AS FECHA,
            TRY_CAST(d.dato AS DECIMAL(18, 8))            AS VALOR,
            CAST(flat.execution_date AS DATE)             AS EXECUTION_DATE,
            flat.filename                                 AS SOURCE_FILE,
            d.fecha                                       AS API_FECHA_RAW
        FROM flat
        LEFT JOIN BANXICO_SERIES bs ON bs.IDSERIE = flat.idserie
    """)

    diag = db.query("""
        SELECT
            COUNT(*) FILTER (WHERE EXISTS (
                SELECT 1 FROM BANXICO_VALUES bv
                WHERE bv.IDSERIE = s.IDSERIE AND bv.FECHA = s.FECHA
            )) AS n_replace,
            COUNT(*) FILTER (WHERE NOT EXISTS (
                SELECT 1 FROM BANXICO_VALUES bv
                WHERE bv.IDSERIE = s.IDSERIE AND bv.FECHA = s.FECHA
            )) AS n_insert,
            COUNT(*) AS n_total
        FROM _BANXICO_VALUES_GAP_STAGE s
    """).iloc[0]
    logger.info(
        "Gap-fill | %d filas en staging: %d REPLACE, %d INSERT",
        int(diag["n_total"]), int(diag["n_replace"]), int(diag["n_insert"]),
    )

    db.command("""
        INSERT OR REPLACE INTO BANXICO_VALUES
            (SECURITY_NAME, IDSERIE, FECHA, VALOR, EXECUTION_DATE, SOURCE_FILE, API_FECHA_RAW)
        SELECT SECURITY_NAME, IDSERIE, FECHA, VALOR, EXECUTION_DATE, SOURCE_FILE, API_FECHA_RAW
        FROM _BANXICO_VALUES_GAP_STAGE
    """)
    logger.info("UPSERT incremental completado")

    silver_dir_incr = (
        DATALAKE / "silver" / "banxico_sie" / "values"
        / f"execution_date={execution_date.isoformat()}"
    )
    silver_dir_incr.mkdir(parents=True, exist_ok=True)
    silver_path_incr = silver_dir_incr / "load=incremental.parquet"
    db.command(f"""
        COPY (
            SELECT bv.*
            FROM BANXICO_VALUES bv
            JOIN _BANXICO_VALUES_GAP_STAGE s
              ON bv.IDSERIE = s.IDSERIE AND bv.FECHA = s.FECHA
        ) TO '{silver_path_incr.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    logger.info("Silver incremental exportado: %s", silver_path_incr)

    summary = db.query("""
        SELECT bv.SECURITY_NAME, bv.IDSERIE,
               COUNT(*)                                       AS N_FILAS,
               MIN(bv.FECHA)                                  AS DESDE,
               MAX(bv.FECHA)                                  AS HASTA,
               SUM(CASE WHEN bv.VALOR IS NULL THEN 1 ELSE 0 END) AS N_NULL
        FROM BANXICO_VALUES bv
        JOIN _BANXICO_VALUES_GAP_STAGE s
          ON bv.IDSERIE = s.IDSERIE AND bv.FECHA = s.FECHA
        GROUP BY bv.SECURITY_NAME, bv.IDSERIE
        ORDER BY bv.SECURITY_NAME
    """)

    db.command("DROP TABLE IF EXISTS _BANXICO_VALUES_GAP_STAGE")

logger.info("Carga incremental finalizada")
summary

In [ ]:
# 2.2 Estadísticas móviles 7 días calendario sobre BANXICO_VALUES
#
# Decisión table-vs-view-vs-function: **VIEW**.
# Justificación:
#   1. CONSISTENCIA: la vista siempre refleja el estado actual de BANXICO_VALUES.
#      Una tabla materializada requeriría lógica de refresh tras cada incremental
#      (riesgo de stale data, idempotencia adicional, otra capa silver).
#   2. VOLUMEN: ~2k filas. La window function es trivialmente rápida en DuckDB.
#      Materializar daría cero beneficio en performance.
#   3. SIMPLICIDAD: una declaración SQL, sin DML extra, sin Parquet silver adicional.
#   4. La elección invertiría con: volumen >> 1M filas, consumers concurrentes con
#      SLA estricto, o stats costosas (percentiles aproximados, t-digest, etc.).
#
# Función UDF descartada: la ventana es fija (7 días), no necesita parametrizarse.
#
# Decisión DuckDB vs Python: 100% DuckDB.
#   - RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW = 7 días CALENDARIO
#     ([FECHA-6, FECHA]), exactamente lo que pide el brief.
#   - Si lo hiciera en pandas tendría que reindexar por fecha calendario por serie
#     (UDIS tiene calendario, los daily hábiles no) -> más código, peor performance.
import logging

logger = logging.getLogger("vmetrix.rolling")
db = get_database()

db.command("""
    CREATE OR REPLACE VIEW BANXICO_VALUES_ROLLING_7D AS
    SELECT
        SECURITY_NAME,
        IDSERIE,
        FECHA                            AS VALUE_DATE,
        VALOR,
        MIN(VALOR) OVER w                AS MIN_7D,
        MAX(VALOR) OVER w                AS MAX_7D,
        AVG(VALOR) OVER w                AS AVG_7D,
        COUNT(VALOR) OVER w              AS N_OBS_7D
    FROM BANXICO_VALUES
    WHERE VALOR IS NOT NULL              -- excluye huecos N/E del cálculo
    WINDOW w AS (
        PARTITION BY IDSERIE
        ORDER BY FECHA
        RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW
    )
""")

n_view = int(db.query("SELECT COUNT(*) AS n FROM BANXICO_VALUES_ROLLING_7D").iloc[0]["n"])
logger.info("Vista BANXICO_VALUES_ROLLING_7D lista: %d filas", n_view)

# --- Verificación: muestra las últimas 5 fechas por security ---
sample = db.query("""
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY SECURITY_NAME ORDER BY VALUE_DATE DESC) AS rn
        FROM BANXICO_VALUES_ROLLING_7D
    )
    SELECT SECURITY_NAME,
           VALUE_DATE,
           VALOR,
           ROUND(MIN_7D, 6) AS MIN_7D,
           ROUND(MAX_7D, 6) AS MAX_7D,
           ROUND(AVG_7D, 6) AS AVG_7D,
           N_OBS_7D
    FROM ranked
    WHERE rn <= 5
    ORDER BY SECURITY_NAME, VALUE_DATE DESC
""")
sample

### 3. Requerimientos transversales del ETL

**3.1 RAW en `s3_datalake`**
Almacene cada respuesta de la API como Parquet antes de transformarla. Diseñe la estructura de carpetas bajo [s3_datalake/](s3_datalake/) (simulando buckets S3). 

**3.2 Idempotencia / reprocesabilidad**
- Correr el pipeline dos veces para el mismo rango **NO** debe generar duplicados.
- El proceso debe ser reprocesable.

**3.3 Logging**
- Use el logger `vmetrix` (ya configurado a INFO por defecto) o cree el suyo.
- Implemente el logging con la profundidad y nivel que estime conveniente para cada tarea del pipeline.

**3.4 DuckDB vs Python**
En cada transformación (parseo de fechas, casting de valores, deduplicación, ventana móvil, merges) indique en un comentario corto por qué eligió DuckDB o Python.

---

### Caso especial `UDIMXN.SPOT`

UDIS **sí** tiene un valor **diario** — no son dos datos al mes. Lo que ocurre es que Banxico publica el valor de UDIS **con anticipación en bloques**:

- Alrededor del **día 10** de cada mes se publica el horizonte de valores diarios hasta el **día 25** siguiente.
- Alrededor del **día 25** de cada mes se publica el horizonte hasta el **día 10** del mes siguiente.

Consecuencias prácticas para el pipeline:
1. `get_last_value('SP68257')` devuelve la fecha del **horizonte publicado** (puede estar varios días en el futuro respecto al runtime), *NO* el valor de hoy.
